# Online Streaming Bootstrap


In a normal bootstrap, we wait until we have the whole dataset, then repeatedly sample from it with replacement. This is fine when we have the whole data.
But in streaming, data continuously coming and this make difficult how to calculate and update standard error for the coming data. 



## 1. Why we use Poisson for streaming bootstrap

In a normal bootstrap, we wait until we have the whole dataset, then repeatedly sample from it **with replacement**.

The problem in streaming data is:

$$
\boxed{\text{we cannot keep all old observations and go back to resample them later}}
$$

So when each observation \(x_i\) arrives, we immediately decide how many times that observation should appear in each bootstrap replicate.

We do this using:

$$
K_{ib}\sim \text{Poisson}(1)
$$

where:

* \(i\) = observation number
* \(b\) = bootstrap replicate
* \(K_{ib}\) = how many times observation \(x_i\) is used in bootstrap \(b\)

For example:

$$
K_{ib}=0
$$

means skip the observation,

$$
K_{ib}=1
$$

means use it once,

$$
K_{ib}=2
$$

means use it twice.

We use \(\lambda=1\) because in an ordinary bootstrap, each original observation appears **once on average**.

The Poisson probability is:

$$
P(K=k)=\frac{e^{-\lambda}\lambda^k}{k!}
$$

and with:

$$
\lambda=1
$$

it becomes:

$$
P(K=k)=\frac{e^{-1}}{k!}
$$

For each bootstrap \(b\), we keep only two running quantities:

$$
S_b^*=\sum_i K_{ib}x_i
$$

and

$$
N_b^*=\sum_i K_{ib}
$$

Then its bootstrap mean is:

$$
\bar X_b^*=\frac{S_b^*}{N_b^*}
$$

Finally, if we have \(B=1000\) bootstrap means, the bootstrap standard error is:

$$
\boxed{
SE=
SD(\bar X_1^*,\bar X_2^*,\ldots,\bar X_{1000}^*)
}
$$

So Poisson solves the main streaming problem: **we can bootstrap observations immediately as they arrive and then discard the raw data.**

---

## 2. What we are going to do in this coding problem

The task gives us:

$$
6 \text{ batches}
$$

Each batch contains:

$$
100,000 \text{ observations}
$$

drawn from:

$$
X\sim Uniform(20,40)
$$

So altogether:

$$
6\times100,000=600,000
$$

observations will arrive.

We want:

$$
B=1000
$$

bootstrap replicates.

At the beginning, we create:

$$
S_1^*,\ldots,S_{1000}^*
$$

and:

$$
N_1^*,\ldots,N_{1000}^*
$$

all initialized to zero.

For every new observation, each of the 1000 bootstraps receives its own:

$$
K\sim Poisson(1)
$$

weight.

Then we update:

$$
S_b^*
\leftarrow
S_b^*+K_{ib}x_i
$$

and:

$$
N_b^*
\leftarrow
N_b^*+K_{ib}
$$

We do this through batch 1. When batch 1 is finished, we **delete its 100,000 raw observations**, but keep the updated \(S_b^*\) and \(N_b^*\).

Then batch 2 arrives and continues updating those **same 1000 bootstrap states**.

The same process continues through all 6 batches.

At the very end, calculate:

$$
\bar X_b^*=\frac{S_b^*}{N_b^*}
$$

for all 1000 bootstrap replicates, and then:

$$
\boxed{
SE=SD(\bar X_1^*,\ldots,\bar X_{1000}^*)
}
$$

For this Uniform\((20,40)\) example, the SE should be approximately:

$$
\boxed{0.0074}
$$

So the coding logic is simply:

$$
\boxed{
\text{generate batch}
\rightarrow
\text{apply Poisson weights}
\rightarrow
\text{update }S,N
\rightarrow
\text{delete batch}
\rightarrow
\text{next batch}
}
$$

and after all 6 batches:

$$
\boxed{
S/N
\rightarrow
1000\text{ bootstrap means}
\rightarrow
SD
\rightarrow
SE
}
$$


In [1]:
import numpy as np

def streaming_bootstrap_se(num_batches=6, batch_size=100000, B=1000):
    
    # information for the 1000 bootstraps
    S = np.zeros(B)   # bootstrap sums
    N = np.zeros(B)   # bootstrap counts

    for batch in range(num_batches):

        
        x = np.random.uniform(20, 40, batch_size)

        # observations one small chunk at a time
        chunk_size = 1000

        for start in range(0, batch_size, chunk_size):
            x_chunk = x[start:start + chunk_size]

            # Each bootstrap gives each observation a Poisson(1) weight
            weights = np.random.poisson(
                1,
                size=(B, len(x_chunk))
            )

            
            S += weights @ x_chunk
            N += weights.sum(axis=1)

        # Deleting this batch before next batch arrives
        del x

    # Mean for each of the 1000 bootstrap streams
    bootstrap_means = S / N

    # Standard error = SD of the bootstrap means
    SE = np.std(bootstrap_means, ddof=1)

    return SE


np.random.seed(42)

se = streaming_bootstrap_se()

print("Bootstrap Standard Error:", se)

Bootstrap Standard Error: 0.007202000239573121
